# Árvore 2-3 e introdução à Árvore B

Este notebook apresenta a **Árvore 2-3** com foco em intuição, implementação e testes, e encerra com uma **introdução prática à Árvore B**.

Objetivos:
- Entender as propriedades estruturais da Árvore 2-3.
- Implementar inserção e busca em Árvore 2-3.
- Validar invariantes com testes automatizados.
- Relacionar Árvore 2-3 com Árvore B.
- Implementar uma versão didática de Árvore B (busca e inserção).

In [12]:
# -*- coding: utf-8 -*-
"""
Seção 1: Imports e configurações básicas.
"""

import random
import time

print("Notebook pronto: Árvore 2-3 e introdução à Árvore B.")

Notebook pronto: Árvore 2-3 e introdução à Árvore B.


## 2) Conceitos da Árvore 2-3

Uma Árvore 2-3 é uma árvore de busca balanceada onde cada nó pode ser:
- **Nó-2**: 1 chave e 2 filhos.
- **Nó-3**: 2 chaves e 3 filhos.

Propriedades:
- Todas as folhas estão no mesmo nível (balanceamento perfeito).
- Chaves em ordem crescente dentro de cada nó.
- Subárvores respeitam os intervalos induzidos pelas chaves do nó.

Com isso, operações como busca e inserção custam tipicamente $O(\log n)$. 

In [13]:
class No23:
    def __init__(self, chaves=None, filhos=None):
        self.chaves = chaves if chaves is not None else []
        self.filhos = filhos if filhos is not None else []

    def folha(self):
        return len(self.filhos) == 0


def indice_descida(chaves, x):
    i = 0
    while i < len(chaves) and x > chaves[i]:
        i += 1
    return i


def inserir_ordenado(lista, x):
    if x in lista:
        return False
    i = indice_descida(lista, x)
    lista.insert(i, x)
    return True

## 3) Inserção em Árvore 2-3 (com divisão de nós)

A inserção ocorre por descida até uma folha e, se houver estouro (3 chaves), fazemos **split**:
- Promovemos a chave do meio para o pai.
- Geramos dois nós filhos com uma chave cada.

Se o pai também estourar, o processo continua para cima.

In [14]:
class Arvore23:
    def __init__(self):
        self.raiz = None
        self.tamanho = 0

    def buscar(self, x):
        no = self.raiz
        while no is not None:
            if x in no.chaves:
                return True
            if no.folha():
                return False
            i = indice_descida(no.chaves, x)
            no = no.filhos[i]
        return False

    def inserir(self, x):
        if self.raiz is None:
            self.raiz = No23(chaves=[x])
            self.tamanho += 1
            return True

        nova_raiz, promocao, inseriu = self._inserir_rec(self.raiz, x)
        if not inseriu:
            return False

        if promocao is not None:
            chave_meio, filho_esq, filho_dir = promocao
            self.raiz = No23(chaves=[chave_meio], filhos=[filho_esq, filho_dir])
        else:
            self.raiz = nova_raiz

        self.tamanho += 1
        return True

    def _split_no_folha(self, no):
        a, b, c = no.chaves
        esquerdo = No23(chaves=[a])
        direito = No23(chaves=[c])
        return b, esquerdo, direito

    def _split_no_interno(self, no):
        k0, k1, k2 = no.chaves
        c0, c1, c2, c3 = no.filhos
        esquerdo = No23(chaves=[k0], filhos=[c0, c1])
        direito = No23(chaves=[k2], filhos=[c2, c3])
        return k1, esquerdo, direito

    def _inserir_rec(self, no, x):
        if no.folha():
            inseriu = inserir_ordenado(no.chaves, x)
            if not inseriu:
                return no, None, False

            if len(no.chaves) <= 2:
                return no, None, True

            chave_meio, esquerdo, direito = self._split_no_folha(no)
            return no, (chave_meio, esquerdo, direito), True

        i = indice_descida(no.chaves, x)
        filho_atualizado, promocao_filho, inseriu = self._inserir_rec(no.filhos[i], x)
        if not inseriu:
            return no, None, False

        if promocao_filho is None:
            no.filhos[i] = filho_atualizado
            return no, None, True

        chave_promovida, filho_esq, filho_dir = promocao_filho
        no.chaves.insert(i, chave_promovida)
        no.filhos[i] = filho_esq
        no.filhos.insert(i + 1, filho_dir)

        if len(no.chaves) <= 2:
            return no, None, True

        chave_meio, esquerdo, direito = self._split_no_interno(no)
        return no, (chave_meio, esquerdo, direito), True

    def em_ordem(self):
        def _percorrer(no):
            if no is None:
                return []
            if len(no.chaves) == 1:
                if no.folha():
                    return [no.chaves[0]]
                return _percorrer(no.filhos[0]) + [no.chaves[0]] + _percorrer(no.filhos[1])

            if no.folha():
                return [no.chaves[0], no.chaves[1]]

            return (
                _percorrer(no.filhos[0])
                + [no.chaves[0]]
                + _percorrer(no.filhos[1])
                + [no.chaves[1]]
                + _percorrer(no.filhos[2])
            )

        return _percorrer(self.raiz)

## 4) Visualização textual da Árvore 2-3

Vamos imprimir a estrutura em níveis para acompanhar as inserções.

In [15]:
def imprimir_23(no, nivel=0, prefixo="Raiz: "):
    if no is None:
        print("(árvore vazia)")
        return

    print("    " * nivel + f"{prefixo}{no.chaves}")
    for i, filho in enumerate(no.filhos):
        imprimir_23(filho, nivel + 1, f"F{i}: ")


arv23 = Arvore23()
sequencia = [40, 20, 60, 10, 30, 50, 70, 25, 27, 26, 65, 80]
for v in sequencia:
    arv23.inserir(v)

print("Árvore 2-3 após inserções:")
imprimir_23(arv23.raiz)
print("Em ordem:", arv23.em_ordem())

Árvore 2-3 após inserções:
Raiz: [40]
    F0: [20, 27]
        F0: [10]
        F1: [25, 26]
        F2: [30]
    F1: [60, 70]
        F0: [50]
        F1: [65]
        F2: [80]
Em ordem: [10, 20, 25, 26, 27, 30, 40, 50, 60, 65, 70, 80]


## 5) Validação de invariantes da Árvore 2-3

Vamos verificar automaticamente:
- Ordem das chaves em cada nó.
- Quantidade válida de filhos (0, 2 ou 3).
- Relação entre número de chaves e número de filhos em nós internos.
- Todas as folhas no mesmo nível.
- Ordenação global (propriedade de busca).

In [16]:
def validar_23(no):
    if no is None:
        return True

    profundidades_folhas = []

    def _validar(no_atual, min_v, max_v, prof):
        if not (1 <= len(no_atual.chaves) <= 2):
            return False

        if len(no_atual.chaves) == 2 and not (no_atual.chaves[0] < no_atual.chaves[1]):
            return False

        for k in no_atual.chaves:
            if not (min_v < k < max_v):
                return False

        if no_atual.folha():
            profundidades_folhas.append(prof)
            return True

        if len(no_atual.filhos) not in (2, 3):
            return False

        if len(no_atual.filhos) != len(no_atual.chaves) + 1:
            return False

        if len(no_atual.chaves) == 1:
            k0 = no_atual.chaves[0]
            return (
                _validar(no_atual.filhos[0], min_v, k0, prof + 1)
                and _validar(no_atual.filhos[1], k0, max_v, prof + 1)
            )

        k0, k1 = no_atual.chaves
        return (
            _validar(no_atual.filhos[0], min_v, k0, prof + 1)
            and _validar(no_atual.filhos[1], k0, k1, prof + 1)
            and _validar(no_atual.filhos[2], k1, max_v, prof + 1)
        )

    ok = _validar(no, -10**18, 10**18, 0)
    if not ok:
        return False

    return len(set(profundidades_folhas)) == 1


assert validar_23(arv23.raiz), "Invariantes da Árvore 2-3 violadas"
print("Validação OK: estrutura 2-3 consistente.")

Validação OK: estrutura 2-3 consistente.


## 6) Teste automatizado com dados aleatórios

Neste teste:
- Inserimos vários valores aleatórios sem repetição.
- Conferimos se `buscar` encontra todos os elementos.
- Verificamos invariantes da estrutura.

In [17]:
random.seed(23)
dados = random.sample(range(1, 20000), 1200)
arv_teste = Arvore23()

for x in dados:
    arv_teste.inserir(x)

for x in random.sample(dados, 200):
    assert arv_teste.buscar(x), f"Falha na busca do valor {x}"

assert validar_23(arv_teste.raiz), "Estrutura inválida após inserções"
em_ordem = arv_teste.em_ordem()
assert em_ordem == sorted(dados), "Percurso em ordem não bate com ordenação esperada"
print("Teste aleatório concluído com sucesso.")

Teste aleatório concluído com sucesso.


## 7) Custo empírico básico na Árvore 2-3

Mediremos tempos de inserção e busca para tamanhos crescentes.

In [ ]:
def benchmark_23(n):
    valores = random.sample(range(1, 20 * n), n)
    consultas = random.sample(valores, min(1000, n))

    arv = Arvore23()

    t0 = time.perf_counter()
    for v in valores:
        arv.inserir(v)
    t_insercao = time.perf_counter() - t0

    t0 = time.perf_counter()
    for q in consultas:
        _ = arv.buscar(q)
    t_busca = time.perf_counter() - t0

    return t_insercao, t_busca


for n in [1000, 3000, 6000, 10000]:
    t_ins, t_bus = benchmark_23(n)
    print(f"n={n:5d} | inserção={t_ins:.6f}s | busca={t_bus:.6f}s")

n= 1000 | insercao=0.002068s | busca=0.001007s
n= 3000 | insercao=0.006598s | busca=0.001144s
n= 6000 | insercao=0.012897s | busca=0.001093s
n=10000 | insercao=0.022207s | busca=0.001078s


## 8) Da Árvore 2-3 para a Árvore B

A Árvore 2-3 é um caso particular da família de árvores multicaminho balanceadas.

Na Árvore B (grau mínimo $t$):
- Cada nó (exceto raiz) guarda entre $t-1$ e $2t-1$ chaves.
- Cada nó interno (exceto raiz) tem entre $t$ e $2t$ filhos.
- Todas as folhas ficam no mesmo nível.

Quando $t=2$, a Árvore B vira uma Árvore 2-3-4.

Ela é muito usada em bancos de dados e sistemas de arquivos, pois reduz acessos a disco ao manter muitos valores por nó.

In [ ]:
class NoB:
    def __init__(self, t, folha=True, chaves=None, filhos=None):
        self.t = t
        self.folha = folha
        self.chaves = chaves if chaves is not None else []
        self.filhos = filhos if filhos is not None else []


class ArvoreB:
    def __init__(self, t=2):
        if t < 2:
            raise ValueError("O grau mínimo t deve ser >= 2")
        self.t = t
        self.raiz = NoB(t=t, folha=True)

    def buscar(self, x, no=None):
        if no is None:
            no = self.raiz

        i = 0
        while i < len(no.chaves) and x > no.chaves[i]:
            i += 1

        if i < len(no.chaves) and x == no.chaves[i]:
            return True

        if no.folha:
            return False

        return self.buscar(x, no.filhos[i])

    def inserir(self, x):
        if self.buscar(x):
            return

        r = self.raiz
        if len(r.chaves) == (2 * self.t - 1):
            s = NoB(t=self.t, folha=False, filhos=[r])
            self._split_filho(s, 0)
            self.raiz = s
            self._inserir_nao_cheio(s, x)
        else:
            self._inserir_nao_cheio(r, x)

    def _split_filho(self, pai, i):
        t = self.t
        y = pai.filhos[i]
        z = NoB(t=t, folha=y.folha)

        chave_meio = y.chaves[t - 1]
        z.chaves = y.chaves[t:]
        y.chaves = y.chaves[:t - 1]

        if not y.folha:
            z.filhos = y.filhos[t:]
            y.filhos = y.filhos[:t]

        pai.filhos.insert(i + 1, z)
        pai.chaves.insert(i, chave_meio)

    def _inserir_nao_cheio(self, no, x):
        i = len(no.chaves) - 1

        if no.folha:
            no.chaves.append(0)
            while i >= 0 and x < no.chaves[i]:
                no.chaves[i + 1] = no.chaves[i]
                i -= 1
            no.chaves[i + 1] = x
            return

        while i >= 0 and x < no.chaves[i]:
            i -= 1
        i += 1

        if len(no.filhos[i].chaves) == (2 * self.t - 1):
            self._split_filho(no, i)
            if x > no.chaves[i]:
                i += 1

        self._inserir_nao_cheio(no.filhos[i], x)

    def altura(self):
        h = 0
        no = self.raiz
        while not no.folha:
            h += 1
            no = no.filhos[0]
        return h

## 9) Demonstração rápida de Árvore B

Vamos usar $t=3$ (cada nó comporta de 2 a 5 chaves, exceto raiz).

In [ ]:
def imprimir_b(no, nivel=0, prefixo="Raiz: "):
    print("    " * nivel + f"{prefixo}{no.chaves}")
    if not no.folha:
        for i, filho in enumerate(no.filhos):
            imprimir_b(filho, nivel + 1, f"F{i}: ")


arvb = ArvoreB(t=3)
for v in [50, 20, 70, 10, 30, 60, 80, 90, 40, 35, 25, 65, 75, 85]:
    arvb.inserir(v)

print("Estrutura da Árvore B (t=3):")
imprimir_b(arvb.raiz)
print("Buscar 65 ->", arvb.buscar(65))
print("Buscar 111 ->", arvb.buscar(111))

Estrutura da Arvore B (t=3):
Raiz: [30, 70]
    F0: [10, 20, 25]
    F1: [35, 40, 50, 60, 65]
    F2: [75, 80, 85, 90]
Buscar 65 -> True
Buscar 111 -> False


## 10) Comparação curta: altura em Árvore 2-3 vs Árvore B

Maior fan-out tende a reduzir altura. Vamos observar isso em um experimento simples.

In [ ]:
def altura_23(no):
    if no is None:
        return -1
    h = 0
    atual = no
    while not atual.folha():
        h += 1
        atual = atual.filhos[0]
    return h


n = 5000
valores = random.sample(range(1, 100000), n)

a23 = Arvore23()
for v in valores:
    a23.inserir(v)

ab2 = ArvoreB(t=2)
ab4 = ArvoreB(t=4)
for v in valores:
    ab2.inserir(v)
    ab4.inserir(v)

print(f"Altura Árvore 2-3: {altura_23(a23.raiz)}")
print(f"Altura Árvore B (t=2): {ab2.altura()}")
print(f"Altura Árvore B (t=4): {ab4.altura()}")

Altura Arvore 2-3: 9
Altura Arvore B (t=2): 9
Altura Arvore B (t=4): 4


## 11) Conclusões

- A Árvore 2-3 oferece balanceamento estrito e boa base conceitual para estruturas multicaminho.
- A Árvore B generaliza a ideia para maior número de chaves por nó, reduzindo altura.
- Em cenários com armazenamento externo (disco), Árvores B e variantes (como B+) são muito utilizadas.

Sugestões de extensão:
1. Implementar remoção completa na Árvore 2-3.
2. Implementar remoção em Árvore B.
3. Comparar com AVL e BST em diferentes distribuições de dados.